In [1]:
import numpy as np
import pandas as pd

seed = 1337
rng = np.random.default_rng(seed)
N = 275000
chunkSize = 10000

columns = ["Global_SKU_Defect_Rate_%", "ABS_Volume_Difference", "Storage_Size", "Aisle_Hold_%", "#_Pick_Events", "#_Pick_Events_In_Clique",
           "#_Picks", "#_Picks_In_Clique", "Defect_In_Linked_Receive", "Time_In_Loc", "Current_Max_Volume"]



problemSKU = [True, False]
problemSKUProbs = np.array([0.05, 0.95])
problemSKUProbsSize1 = np.array([0.1, 0.9])



#Parameters 
#Global_SKU_Defect_Rate_%
#Beta distribution 
defectAlpha, defectBeta = 2, 100

#ABS_Volume_Difference
#Normal distribution 
absVolumeMean = 0
absVolumeStd = 1000

#Storage_Size
#Assigned Distribution
storageSizes = ['size_1', 'size_2', 'size_3', 'size_4']
storageSizeProbs = np.array([0.4, 0.2, 0.3, 0.1])
#rng.choice(storageSizes, size=1000, p=storageSizeProbs)

#Aisle_Hold_%
#Beta distribution
aisleHoldAlpha, aisleHoldBeta = 0.425, 8.075

##_Pick_Events
#Poisson distribution
pickEventsLambda = 15

##_Pick_Events_In_Clique
#Poisson distribution
pickEventsInCliqueLambda = 100

##_Picks
#Poisson distribution
picksLambda = 35

##_Picks_In_Clique
#Poisson distribution
picksInCliqueLambda = 200

#Defect_In_Linked_Receive
#Assigned Distribution
defectInLinkedReceive = [True, False]
defectInLinkedReceiveProbs = np.array([0.05, 0.95])
defectInLinkedReceiveProbsIfProbSKU = np.array([0.15, 0.85])

#Time_In_Loc
#Negative Binomial distribution
timeInLocAlphaN = 9
timeInLocBetaP = 0.35

#Current_Max_Volume
#Normal distribution
currentMaxVolumeMean = 2160
currentMaxVolumeStd = 600

parameters_to_save = {
    "seed": seed,
    "problemSKUProbs": problemSKUProbs.tolist(),
    "problemSKUProbsSize1": problemSKUProbsSize1.tolist(),
    "defectAlpha": defectAlpha,
    "defectBeta": defectBeta,
    "absVolumeMean": absVolumeMean,
    "absVolumeStd": absVolumeStd,
    "storageSizes": storageSizes,
    "storageSizeProbs": storageSizeProbs.tolist(),
    "aisleHoldAlpha": aisleHoldAlpha,
    "aisleHoldBeta": aisleHoldBeta,
    "pickEventsLambda": pickEventsLambda,
    "pickEventsInCliqueLambda": pickEventsInCliqueLambda,
    "picksLambda": picksLambda,
    "picksInCliqueLambda": picksInCliqueLambda,
    "defectInLinkedReceiveProbs": defectInLinkedReceiveProbs.tolist(),
    "defectInLinkedReceiveProbsIfProbSKU": defectInLinkedReceiveProbsIfProbSKU.tolist(),
    "timeInLocAlphaN": timeInLocAlphaN,
    "timeInLocBetaP": timeInLocBetaP,
    "currentMaxVolumeMean": currentMaxVolumeMean,
    "currentMaxVolumeStd": currentMaxVolumeStd,
}

In [2]:
import inspect

def apply_problem_effects(arr, problem_mask, rng,
                          p_problem=0.6, p_nonproblem=0.1,
                          transform=None):
    """
    For each element, with prob p_problem if problem SKU, or p_nonproblem otherwise,
    apply 'transform' to that element. Returns modified copy.
    """
    arr = np.array(arr, copy=True)
    n = arr.shape[0]

    # Base random mask
    base_mask = rng.random(n)

    mask_problem = problem_mask & (base_mask < p_problem)
    mask_nonproblem = (~problem_mask) & (base_mask < p_nonproblem)

    mask = mask_problem | mask_nonproblem
    if transform is not None and mask.any():
        arr[mask] = transform(arr[mask], rng)
    return arr

def make_chunk(chunkSize, rng):
    # ----- 1. base covariates -----

    # Storage_Size
    storage = rng.choice(storageSizes, size=chunkSize, p=storageSizeProbs)

    # Problem SKU mask
    problem_mask = rng.choice(
        problemSKU, size=chunkSize, p=problemSKUProbs
    )

    size_1_mask = (storage == "size_1")
    problem_mask[size_1_mask] = rng.choice(
        problemSKU, size=size_1_mask.sum(), p=problemSKUProbsSize1
    )

    # Global_SKU_Defect_Rate_% (as fraction 0–1)
    global_defect = rng.beta(defectAlpha, defectBeta, size=chunkSize)

    # ABS_Volume_Difference
    abs_vol = np.abs(rng.normal(absVolumeMean, absVolumeStd, size=chunkSize))

    # Aisle_Hold_% (0–1)
    aisle_hold = rng.beta(aisleHoldAlpha, aisleHoldBeta, size=chunkSize)

    # Poisson counts
    pick_events = rng.poisson(pickEventsLambda, size=chunkSize)
    pick_events_clique = rng.poisson(pickEventsInCliqueLambda, size=chunkSize)
    picks = rng.poisson(picksLambda, size=chunkSize)
    picks_clique = rng.poisson(picksInCliqueLambda, size=chunkSize)

    # Defect_In_Linked_Receive (True/False)
    defect_linked = rng.choice(
        [True, False], size=chunkSize, p=defectInLinkedReceiveProbs
    )
    defect_linked_mask = (defect_linked == True)
    problem_mask[defect_linked_mask] = rng.choice(
        problemSKU, size=defect_linked_mask.sum(), p=defectInLinkedReceiveProbsIfProbSKU
    )
    
    # Time_In_Loc ~ NB(n, p)
    time_in_loc = rng.negative_binomial(timeInLocAlphaN, timeInLocBetaP, size=chunkSize)
    time_in_loc = time_in_loc + 1  # avoid log(0)

    # Current_Max_Volume (non-negative)
    current_max_vol = np.abs(
        rng.normal(currentMaxVolumeMean, currentMaxVolumeStd, size=chunkSize)
    )

    """ 
    size_1_mask = (storage == "size_1")  # adjust to your label
    # Boost base global defect a bit for size_1, but softly
    global_defect[size_1_mask] = np.clip(
        global_defect[size_1_mask] * rng.uniform(1.2, 1.6, size=size_1_mask.sum()),
        0,
        1,
    )
    
    """
    # ----- 2. carton-size–specific defect behavior (size_1) -----

    

    # ----- 3. apply subtle problem effects with helper -----

    # Global defect: multiplicative bump + cap
    global_defect = apply_problem_effects(
        global_defect, problem_mask, rng,
        p_problem=0.7, p_nonproblem=0.1,
        transform=lambda x, r: np.clip(
            x * r.uniform(1.3, 1.8, size=x.shape[0]), 0, 1
        ),
    )

    # ABS volume: right shift
    abs_vol = apply_problem_effects(
        abs_vol, problem_mask, rng,
        p_problem=0.6, p_nonproblem=0.02,
        transform=lambda x, r: x * r.uniform(1.1, 2.3, size=x.shape[0])
    )

    # Aisle hold: additive bump
    aisle_hold = apply_problem_effects(
        aisle_hold, problem_mask, rng,
        p_problem=0.6, p_nonproblem=0.15,
        transform=lambda x, r: np.clip(
            x + r.beta(1.5, 8, size=x.shape[0]), 0, 1
        ),
    )

    # Pick events: extra demand
    pick_events = apply_problem_effects(
        pick_events, problem_mask, rng,
        p_problem=0.5, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(.9, 1.3, size=x.shape[0])
    )

    pick_events_clique = apply_problem_effects(
        pick_events_clique, problem_mask, rng,
        p_problem=0.5, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(.8, 1.4, size=x.shape[0]),
    )

    picks = apply_problem_effects(
        picks, problem_mask, rng,
        p_problem=0.5, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(1.1, 1.5, size=x.shape[0]),
    )

    picks_clique = apply_problem_effects(
        picks_clique, problem_mask, rng,
        p_problem=0.5, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(1.3, 1.8, size=x.shape[0]),
    )

    time_in_loc = apply_problem_effects(
        time_in_loc, problem_mask, rng,
        p_problem=0.5, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(1.0, 1.2, size=x.shape[0]),
    )

    current_max_vol = apply_problem_effects(
        current_max_vol, problem_mask, rng,
        p_problem=0.4, p_nonproblem=0.1,
        transform=lambda x, r: x * r.uniform(0.3, 0.6, size=x.shape[0]),
    )

# ----- 5. build 2D numeric matrix, standardize -----

    numeric_cols = np.column_stack([
        global_defect * 100.0,       # Global_SKU_Defect_Rate_%
        abs_vol,                     # ABS_Volume_Difference
        aisle_hold * 100.0,          # Aisle_Hold_%
        pick_events,                 # #_Pick_Events
        pick_events_clique,          # #_Pick_Events_In_Clique
        picks,                       # #_Picks
        picks_clique,                # #_Picks_In_Clique
        time_in_loc,                 # Time_In_Loc
        current_max_vol,             # Current_Max_Volume
    ])
    """ 
    # NEW: Min-Max Normalization to [0, 1]
    mins = numeric_cols.min(axis=0)
    maxs = numeric_cols.max(axis=0)
    ranges = maxs - mins
    ranges_safe = np.where(ranges == 0, 1, ranges)
    numeric_norm = (numeric_cols - mins) / ranges_safe

    # Unpack normalized columns (same names, but now [0,1] scaled)
    (
    global_defect_norm,      # norm
    abs_vol_norm,
    aisle_hold_norm,
    pick_events_norm,
    pick_events_clique_norm,
    picks_norm,
    picks_clique_norm,
    time_in_loc_norm,
    current_max_vol_norm,
        ) = numeric_norm.T
    """
    means = numeric_cols.mean(axis=0)
    stds = numeric_cols.std(axis=0, ddof=0)
    stds_safe = np.where(stds == 0, 1, stds)
    numeric_std = (numeric_cols - means) / stds_safe

    # Unpack standardized columns for readability
    (
        global_defect_std,
        abs_vol_std,
        aisle_hold_std,
        pick_events_std,
        pick_events_clique_std,
        picks_std,
        picks_clique_std,
        time_in_loc_std,
        current_max_vol_std,
    ) = numeric_std.T
    # One-hot for Storage_Size (drop one level to avoid perfect collinearity)
    size_1 = (storage == "size_1").astype(float)
    size_2 = (storage == "size_2").astype(float)
    size_3 = (storage == "size_3").astype(float)
    # baseline: size_4 when all above == 0

    # Binary 0/1 for Defect_In_Linked_Receive
    defect_linked_num = defect_linked.astype(float)

    # ----- 6. logistic coefficients on standardized scale -----

    a0 = -.75           # intercept
    b_global = .3
    b_abs_vol = .6
    b_aisle = 0.3
    b_pick_events = 0.2
    b_pick_clique = 0.3
    b_picks = 0.2
    b_picks_clique = 0.3
    b_time = 0.4
    b_max_vol = -0.1

    # New categorical coefficients (on standardized or raw 0/1 scale)
    b_size_1 = 0.35
    b_size_2 = 0.15
    b_size_3 = 0.1
    b_defect_linked = 0.8

    logit_p = (
    a0
    + b_global * global_defect_std  
    + b_abs_vol * abs_vol_std
    + b_aisle * aisle_hold_std
    + b_pick_events * pick_events_std
    + b_pick_clique * pick_events_clique_std
    + b_picks * picks_std
    + b_picks_clique * picks_clique_std
    + b_time * time_in_loc_std
    + b_max_vol * current_max_vol_std
    + b_size_1 * size_1
    + b_size_2 * size_2
    + b_size_3 * size_3
    + b_defect_linked * defect_linked_num
        )

    p_defect = np.clip(1 / (1 + np.exp(-logit_p)), 0.001, 0.999)
    #defect = rng.binomial(1, p_defect)

    # ----- 7. build DataFrame from standardized numeric + categoricals -----
    """"
     df_chunk = pd.DataFrame(
        numeric_norm    ,  # was numeric_std
        columns=[
            "Global_SKU_Defect_Rate_%_norm",  # was _std
            "ABS_Volume_Difference_norm",
            "Aisle_Hold_%_norm",
            "#_Pick_Events_norm",
            "#_Pick_Events_In_Clique_norm",
            "#_Picks_norm",
            "#_Picks_In_Clique_norm",
            "Time_In_Loc_norm",
            "Current_Max_Volume_norm",
        ],
    )   
    """
    df_chunk = pd.DataFrame(
        numeric_std,
        columns=[
            "Global_SKU_Defect_Rate_%_std",
            "ABS_Volume_Difference_std",
            "Aisle_Hold_%_std",
            "#_Pick_Events_std",
            "#_Pick_Events_In_Clique_std",
            "#_Picks_std",
            "#_Picks_In_Clique_std",
            "Time_In_Loc_std",
            "Current_Max_Volume_std",
        ],
    )

    df_chunk["Storage_Size"] = storage
    df_chunk["Defect_In_Linked_Receive"] = defect_linked
    df_chunk["Problem_SKU"] = problem_mask
    #df_chunk["Defect"] = defect
    df_chunk["Logit_Probability"] = logit_p
    df_chunk["True_Prob_Defect"] = p_defect

    return df_chunk

funcs_to_save = {
    "apply_problem_effects": inspect.getsource(apply_problem_effects),
    "make_chunk": inspect.getsource(make_chunk),
}

In [3]:


import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
#out_path = os.getenv("DATA_SAVE_DIR", "no_env_set")
data_dir = f"./SyntheticData/{timestamp}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
out_path = f"{data_dir}/synthetic_data_{timestamp}.csv"
print(out_path)
first = True
remaining = N

while remaining > 0:
    n_now = min(chunkSize, remaining)
    df = make_chunk(n_now, rng)

    df.to_csv(
        out_path,
        mode="w" if first else "a",
        header=first,
        index=False,
    )
    first = False
    remaining -= n_now

./SyntheticData/2026_03_29_16_42_05/synthetic_data_2026_03_29_16_42_05.csv


In [4]:
import pandas as pd
import numpy as np
import json

all_parameters = parameters_to_save | funcs_to_save
with open(os.path.join(data_dir, f"{timestamp}_generation_parameters.json"), "w", encoding="utf-8") as f:
    json.dump(all_parameters, f, indent=2)

# Load your most recent synthetic data
df = pd.read_csv(out_path)  # replace with your filename

import plotly.express as px
import plotly.graph_objects as go

numeric_cols = ["Global_SKU_Defect_Rate_%_std",
                "ABS_Volume_Difference_std",
                "Aisle_Hold_%_std",
                "#_Pick_Events_std",
                "#_Pick_Events_In_Clique_std",
                "#_Picks_std",
                "#_Picks_In_Clique_std",
                "Time_In_Loc_std",
                "Current_Max_Volume_std"
            ] 
for col in numeric_cols:

    fig = px.histogram(df, x=col, color='Problem_SKU',
                       nbins=30, opacity=0.7,
                       histnorm='probability density',
                       barmode='overlay',
                       title=f'Feature Distributions by Defect Status - {col}')
    plot_path = os.path.join(data_dir, f"{col}_{timestamp}_grouped_histogram.png")
    meta_path = os.path.join(data_dir, f"{col}.png.meta.json")
    fig.write_image(plot_path)   
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "caption": f"Feature Distributions by Defect Status - {col}",
                "description": f"Overlay histogram of {col} grouped by Problem_SKU with probability density normalization.",
            },
            f,
            indent=2,
        ) 
    #fig.show()

for col in numeric_cols:
    fig = px.histogram(df, x=col, color='Problem_SKU', 
                       facet_col='Problem_SKU', facet_col_wrap=2,
                       nbins=30, opacity=0.7,
                       histnorm='probability density',
                       title=f'Feature Distributions by Defect Status - {col}')
    plot_path = os.path.join(data_dir, f"{col}_{timestamp}_histogram.png")
    meta_path = os.path.join(data_dir, f"{col}.png.meta.json")
    fig.write_image(plot_path)   
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "caption": f"Feature Distributions by Defect Status - {col}",
                "description": f"Overlay histogram of {col} grouped by Problem_SKU with probability density normalization.",
            },
            f,
            indent=2,
        )
    #fig.show()
